# finder

> Pick the right skills for a request: embedding similarity when you hand it an embedder (kosha's
> `static_embedder` slots straight in), lexical overlap otherwise, with an optional
> `chat.classify` confirmation pass.

In [ ]:
#| default_exp finder

In [ ]:
#| hide
from nbdev.showdoc import *

Small on-device models can't reliably pick from a long catalog in-prompt, so selection happens
outside the conversation: rank every skill's description + triggers against the request, keep the
top `k` above a floor, and (optionally) let the model veto each survivor with a cheap isolated
`classify` call -- rishi's `classify` runs in a throwaway conversation on the same engine, so
confirmation never pollutes the live KV cache.

In [ ]:
#| export
import re, math
from fastcore.utils import store_attr

In [ ]:
#| export
def _tokens(s): return set(re.findall(r'[a-z0-9]+', s.lower()))

def lexical_score(request:str, skill) -> float:
    'Fraction of request tokens found in the skill\'s name/description/triggers; +1 if the name is mentioned.'
    req = _tokens(request)
    if not req: return 0.
    hay = _tokens(f"{skill.name} {skill.description} {' '.join(skill.triggers)}")
    score = len(req & hay) / len(req)
    if skill.name.lower() in request.lower(): score += 1.
    return score

In [ ]:
#| export
def _cos(a, b):
    num = sum(x*y for x,y in zip(a,b))
    den = math.sqrt(sum(x*x for x in a)) * math.sqrt(sum(y*y for y in b))
    return num/den if den else 0.

In [ ]:
#| export
class Finder:
    'Rank a `Registry`\'s skills against a request and pick the top candidates.'
    def __init__(self,
                 registry,          # the Registry to pick from
                 embed=None,        # embed(list[str])->list[vector]; None = lexical scoring
                 chat=None,         # rishi Chat used only when pick(confirm=True)
                 thresh:float=0.05):# minimum score to be considered at all
        store_attr()
    def rank(self, request:str):
        'All skills as `(skill, score)`, best first.'
        sks = list(self.registry)
        if not sks: return []
        if self.embed is not None:
            docs = [f"{s.name}: {s.one_line()} {' '.join(s.triggers)}" for s in sks]
            vecs, q = self.embed(docs), self.embed([request])[0]
            scores = [_cos(q, v) for v in vecs]
        else: scores = [lexical_score(request, s) for s in sks]
        return sorted(zip(sks, scores), key=lambda t: -t[1])
    def pick(self, request:str, k:int=2, confirm:bool=False):
        'Top `k` skills above `thresh`; with `confirm=True` and a chat, each survivor is model-vetted.'
        cands = [s for s,sc in self.rank(request)[:k] if sc >= self.thresh]
        if confirm and self.chat is not None:
            cands = [s for s in cands
                     if self.chat.classify(f'Task: {request}\nSkill "{s.name}": {s.one_line()}',
                                           ['useful for this task', 'not useful']) == 'useful for this task']
        return cands

In [ ]:
from ramabana.registry import Skill

class _Reg(list):
    skills = property(lambda self: self)

sks = _Reg([Skill('fossick','py','web search, page fetching, crawling, and browser automation','fossick.skill',
                  triggers=['about to call WebSearch']),
            Skill('kosha','py','search your repo and installed packages before writing new code','kosha.skill'),
            Skill('demo','md','a demo skill','demo/SKILL.md')])
f = Finder(sks)
top = f.pick('search the web for litert docs and fetch the page', k=2)
assert top and top[0].name == 'fossick'
assert f.pick('what existing code in my repo already parses frontmatter?', k=1)[0].name == 'kosha'
assert f.pick('zzz qqq', k=3) == []   # nothing relevant -> nothing loaded

In [ ]:
# embedding path: any embed(list[str])->vectors works; here a toy bag-of-words embedder
def bow(texts):
    vocab = ['web','search','repo','code','demo','fetch']
    return [[t.lower().count(w) for w in vocab] for t in texts]
fe = Finder(sks, embed=bow, thresh=0.1)
assert fe.pick('web search and fetch', k=1)[0].name == 'fossick'

In [ ]:
# confirm path uses chat.classify's first label as the positive
class _FakeChat:
    def classify(self, text, labels): return labels[0] if 'fossick' in text else labels[1]
fc = Finder(sks, chat=_FakeChat())
names = [s.name for s in fc.pick('search the web for litert docs', k=2, confirm=True)]
assert names == ['fossick']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()